# NumCompute-Stream: Streaming ML Demo

Demonstrates the streaming ML framework built on top of NumCompute.

**Covers:**
- Loading CSV data via `io.py`
- Splitting into chunks to simulate a data stream
- Incremental training with `DecisionTreeClassifier` and `EnsembleClassifier`
- Per-chunk metric logging via `StreamTrainer`
- Visualisation using `visualise.py`
- `StreamingMetrics` for cumulative and rolling-window evaluation
- `Pipeline.partial_fit()` chaining scaler and model


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend

from numcompute_stream import io, visualise
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.ensemble import EnsembleClassifier
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.preprocessing import StandardScaler
from numcompute_stream.metrics import StreamingMetrics
from numcompute_stream.pipeline import Pipeline

print('All modules loaded successfully')


All modules loaded successfully


## 1. Load Data using io.py

In [2]:
data, header = io.load_csv('employees.csv')
print('Headers:', header)
print('Shape:  ', data.shape)
print('First 3 rows:')
for row in data[:3]:
    print(' ', row)


Headers: [['E1000' 'HR' 'Melbourne' 'Junior' '6' 'nan' '3.3' '38.1' '7.4']
 ['E1001' 'Sales' 'Brisbane' 'Mid' '14' '123301' '2.6' '31.2' '6.7']
 ['E1002' 'Finance' 'Brisbane' 'Mid' '0' '95787' '2.6' '41.6' '6.8']
 ['E1003' 'Marketing' 'Melbourne' 'Mid' 'nan' '135448' 'nan' '48.6' '6.3']
 ['E1004' 'Engineering' 'Perth' 'Mid' '9' '122005' '3.3' '32.3' '7.8']
 ['E1005' 'Finance' 'Melbourne' 'Mid' '2' '95213' '1.6' '44.3' '7.7']
 ['E1006' 'HR' 'Adelaide' 'Mid' '5' '102100' '1.8' '38.4' '6.0']
 ['E1007' 'HR' 'Brisbane' 'Junior' '9' '82930' '5.0' '39.7' '6.9']
 ['E1008' 'Engineering' 'Melbourne' 'Junior' '5' '93786' '2.0' '34.7'
  '8.8']
 ['E1009' 'Sales' 'Brisbane' 'Lead' '1' '171966' '2.1' '38.3' '6.4']
 ['E1010' 'Engineering' 'Melbourne' 'Lead' '11' 'nan' '4.9' '46.5' '5.8']
 ['E1011' 'Finance' 'Melbourne' 'Mid' '12' '112555' '5.0' '42.9' '7.0']
 ['E1012' 'Marketing' 'Adelaide' 'Junior' '2' '78207' '2.1' '48.7' '7.3']
 ['E1013' 'HR' 'Melbourne' 'Lead' '5' '178289' '2.7' '45.9' '6.7']
 ['E

## 2. Create Streaming Dataset (Synthetic)

In [3]:
rng = np.random.default_rng(42)
n_samples, n_features = 500, 6

X_full = rng.standard_normal((n_samples, n_features))
w      = rng.standard_normal(n_features)
y_full = (X_full @ w > 0).astype(int)

CHUNK_SIZE = 50
chunks = [
    (X_full[i:i+CHUNK_SIZE], y_full[i:i+CHUNK_SIZE])
    for i in range(0, n_samples, CHUNK_SIZE)
]

print(f'Total samples : {n_samples}')
print(f'Chunk size    : {CHUNK_SIZE}')
print(f'Number chunks : {len(chunks)}')
print(f'Class counts  : {np.bincount(y_full)}')


Total samples : 500
Chunk size    : 50
Number chunks : 10
Class counts  : [262 238]


## 3. Incremental Training: Decision Tree

In [4]:
tree_trainer = StreamTrainer(
    model=DecisionTreeClassifier(max_depth=5, criterion='gini')
)

for i, (Xc, yc) in enumerate(chunks):
    tree_trainer.fit_chunk(Xc, yc)
    e = tree_trainer.log_[-1]
    print(f"Chunk {i+1:2d} | n={e['n_samples']:3d} | "
          f"acc={e['accuracy']:.4f} | "
          f"mem={e['memory_bytes']:6d}B | "
          f"time={e['fit_time_s']:.4f}s")


Chunk  1 | n= 50 | acc=1.0000 | mem=  2800B | time=0.0196s
Chunk  2 | n= 50 | acc=1.0000 | mem=  5600B | time=0.0422s
Chunk  3 | n= 50 | acc=1.0000 | mem=  8400B | time=0.0746s


Chunk  4 | n= 50 | acc=0.9400 | mem= 11200B | time=0.1270s
Chunk  5 | n= 50 | acc=1.0000 | mem= 14000B | time=0.1477s


Chunk  6 | n= 50 | acc=0.9800 | mem= 16800B | time=0.1860s


Chunk  7 | n= 50 | acc=0.9400 | mem= 19600B | time=0.2304s


Chunk  8 | n= 50 | acc=0.9600 | mem= 22400B | time=0.2672s


Chunk  9 | n= 50 | acc=0.9000 | mem= 25200B | time=0.2970s


Chunk 10 | n= 50 | acc=0.9600 | mem= 28000B | time=0.3335s


## 4. Incremental Training: Random Forest

In [5]:
forest_trainer = StreamTrainer(
    model=EnsembleClassifier(
        n_estimators=10,
        method='random_forest',
        max_depth=5,
        random_state=42
    )
)

for Xc, yc in chunks:
    forest_trainer.fit_chunk(Xc, yc)

print('Random Forest training complete')
cm = forest_trainer.cumulative_metrics()
print(f"Final accuracy : {cm['accuracy']:.4f}")
print(f"Precision macro: {cm['precision_macro']:.4f}")
print(f"Recall macro   : {cm['recall_macro']:.4f}")
print(f"F1 macro       : {cm['f1_macro']:.4f}")


Random Forest training complete
Final accuracy : 0.9320
Precision macro: 0.9348
Recall macro   : 0.9303
F1 macro       : 0.9316


## 5. Visualise Streaming Accuracy

In [6]:
tree_acc   = tree_trainer.accuracy_history()
forest_acc = forest_trainer.accuracy_history()

visualise.plot_metric_over_time(
    tree_acc,
    title='Decision Tree Accuracy over Chunks',
    ylabel='Accuracy',
    show=False,
    save_path='tree_accuracy.png'
)
print('Saved: tree_accuracy.png')


Saved: tree_accuracy.png


In [7]:
visualise.compare_models(
    tree_acc, forest_acc,
    labels=('Decision Tree', 'Random Forest'),
    title='Model Comparison: Streaming Accuracy',
    ylabel='Accuracy',
    show=False,
    save_path='model_comparison.png'
)
print('Saved: model_comparison.png')


Saved: model_comparison.png


## 6. Predictions vs Ground Truth (Final Chunk)

In [8]:
X_last, y_last = chunks[-1]

tree_preds   = tree_trainer.model.predict(X_last)
forest_preds = forest_trainer.model.predict(X_last)

visualise.plot_predictions_vs_ground_truth(
    y_last, tree_preds,
    title='Decision Tree: Final Chunk Predictions',
    show=False,
    save_path='tree_predictions.png'
)
print('Saved: tree_predictions.png')

visualise.plot_predictions_vs_ground_truth(
    y_last, forest_preds,
    title='Random Forest: Final Chunk Predictions',
    show=False,
    save_path='forest_predictions.png'
)
print('Saved: forest_predictions.png')


Saved: tree_predictions.png


Saved: forest_predictions.png


## 7. StreamingMetrics: Cumulative & Rolling

In [9]:
sm = StreamingMetrics(window_size=3)

for Xc, yc in chunks:
    preds = tree_trainer.model.predict(Xc)
    sm.update(yc, preds)

result = sm.result()
print('=== Cumulative Metrics ===')
for k, v in result.items():
    if k != 'confusion_matrix':
        print(f'  {k}: {v}')

rolling = sm.rolling_result()
print('\n=== Rolling Window (last 3 chunks) ===')
for k, v in rolling.items():
    if k != 'confusion_matrix':
        print(f'  {k}: {v}')


=== Cumulative Metrics ===
  accuracy: 0.946
  precision_macro: 0.9461661983139302
  recall_macro: 0.9455866315992045
  f1_macro: 0.9458420754921351
  n_chunks: 10

=== Rolling Window (last 3 chunks) ===
  accuracy: 0.9466666666666667
  precision_macro: 0.9466571834992887
  recall_macro: 0.9466571834992887
  f1_macro: 0.9466571834992887
  n_chunks: 3


## 8. Pipeline: StandardScaler + DecisionTree (streaming)

In [10]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  DecisionTreeClassifier(max_depth=4)),
])

for Xc, yc in chunks:
    pipe.partial_fit(Xc, yc)

final_preds = pipe.predict(X_full)
final_acc   = float(np.mean(final_preds == y_full))
print(f'Pipeline final accuracy: {final_acc:.4f}')


Pipeline final accuracy: 0.8920


## Summary

| Module | Streaming Feature |
|--------|-------------------|
| `io.py` | CSV loading with NaN handling |
| `preprocessing.py` | `StandardScaler.partial_fit()` |
| `tree.py` | `DecisionTreeClassifier.partial_fit()` |
| `ensemble.py` | `EnsembleClassifier.partial_fit()` |
| `stream.py` | `StreamTrainer` — per-chunk logging |
| `metrics.py` | `StreamingMetrics` — cumulative & rolling |
| `visualise.py` | `plot_metric_over_time`, `compare_models` |
| `pipeline.py` | `Pipeline.partial_fit()` |
